<a href="https://colab.research.google.com/github/angeshass/PoPE-vs-RoPE/blob/main/%D0%BA%D1%83%D1%80%D1%81%D0%BE%D0%B2%D0%B0%D1%8F.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers datasets accelerate

In [ ]:
!pip install -q datasets evaluate transformers accelerate

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments
from datasets import load_dataset
import math

In [ ]:
model_name = "timinar/baby-llama-58m"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model_rope = AutoModelForCausalLM.from_pretrained(model_name)

In [ ]:
dataset = load_dataset("wikitext", "wikitext-2-raw-v1")

def tokenize(example):
    return tokenizer(example["text"], truncation=True, max_length=1024)

tokenized = dataset.map(tokenize, batched=True, remove_columns=["text"])

def filter_empty(example):
    return len(example["input_ids"]) > 0

tokenized = tokenized.filter(filter_empty)

small_train = tokenized["train"].select(range(5000))
small_val = tokenized["validation"].select(range(1000))

In [ ]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [ ]:
import csv
import time

class GPUStatsCallback(TrainerCallback):
    def __init__(self, filename="gpu_log.csv"):
        self.filename = filename

        with open(self.filename, "w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(["step", "allocated_GB", "reserved_GB", "max_allocated_GB", "time"])

    def on_log(self, args, state, control, logs=None, **kwargs):
        if torch.cuda.is_available():
            allocated = torch.cuda.memory_allocated() / 1024**3
            reserved = torch.cuda.memory_reserved() / 1024**3
            max_allocated = torch.cuda.max_memory_allocated() / 1024**3

            with open(self.filename, "a", newline="") as f:
                writer = csv.writer(f)
                writer.writerow([
                    state.global_step,
                    allocated,
                    reserved,
                    max_allocated,
                    time.time()
                ])

In [ ]:
from transformers import TrainerCallback
import torch
import time


training_args = TrainingArguments(
    output_dir="./rope",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=1,
    num_train_epochs=2,
    logging_steps=50,
    save_steps=500,
)

trainer_rope = Trainer(
    model=model_rope,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_val,
    data_collator=data_collator,
    callbacks=[GPUStatsCallback()]
)

start = time.perf_counter()

trainer_rope.train()

end = time.perf_counter()

print((end-start)/60)


In [ ]:
eval_rope = trainer_rope.evaluate()
ppl_rope = math.exp(eval_rope["eval_loss"])

print("RoPE perplexity:", ppl_rope)

In [ ]:
print("Eval res rope:", eval_rope)

In [ ]:
dataset = load_dataset("cais/mmlu", "abstract_algebra", split="test")

In [ ]:
def format_question(model, example):
    question = example["question"]
    choices = example["choices"]

    prompt = question + "\n"
    for i, choice in enumerate(choices):
        prompt += f"{chr(65+i)}. {choice}\n"

    prompt += "Answer:"
    return prompt, example["answer"]

def get_answer(model, prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=5
        )

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return text


def evaluate_mmlu(model, dataset, n_samples=50):
    correct = 0

    for i in range(n_samples):
        prompt, label = format_question(model, dataset[i])
        output = get_answer(model, prompt)

        predicted = output.strip()[-1]  # последняя буква

        if predicted in ["A", "B", "C", "D"]:
            pred_idx = ord(predicted) - ord("A")
            if pred_idx == label:
                correct += 1

    acc = correct / n_samples
    print(f"Accuracy: {acc:.2f}")

    return acc

In [ ]:
rope_acc = evaluate_mmlu(model_rope, dataset)

# PoPE

In [ ]:
model_pope = AutoModelForCausalLM.from_pretrained(model_name)

In [ ]:
import torch
from transformers.models.llama import modeling_llama

def rotate_half(x):
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)

def apply_pope(q, k, cos, sin, position_ids):

    if position_ids is None:
        position_ids = torch.arange(q.shape[-2], device=q.device).unsqueeze(0)

    pos = position_ids.float()

    r = torch.log(pos + 1).unsqueeze(-1)
    theta = pos / 10000

    cos = torch.cos(theta).unsqueeze(-1)
    sin = torch.sin(theta).unsqueeze(-1)

    q = r * (q * cos + rotate_half(q) * sin)
    k = r * (k * cos + rotate_half(k) * sin)

    return q, k

original_apply_rope = modeling_llama.apply_rotary_pos_emb

def patched_apply_rotary_pos_emb(q, k, cos, sin, position_ids=None):
    return apply_pope(q, k, cos, sin, position_ids)

modeling_llama.apply_rotary_pos_emb = patched_apply_rotary_pos_emb

In [ ]:
training_args_pope = TrainingArguments(
    output_dir="./pope",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=1,
    num_train_epochs=2,
    logging_steps=50,
    save_steps=500,
)

trainer_pope = Trainer(
    model=model_pope,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_val,
    data_collator=data_collator,
    callbacks=[GPUStatsCallback()]
)

trainer_pope.train()

In [ ]:
pope_acc = evaluate_mmlu(model_pope, dataset)